# osapiens Makeathon 2026 - Final Submission Pipeline

Current best:
- Union IoU: 54.23%
- Recall: 79.4%
- FPR: 36.9%
- Year: 0.0%

Best model family so far:
- all-year raw AEF features
- LightGBM / XGBoost tabular model
- threshold around 0.15-0.18


## Execution Order

1. Config
2. Helpers
3. Build or load pseudo-GT
4. Build training matrix
5. Train LightGBM final
6. Predict LightGBM probabilities
7. Train or load XGBoost
8. Predict XGBoost probabilities
9. Make threshold submissions
10. Make ensemble submissions
11. Save feature importance and run metadata

Leaderboard notes stay in markdown only. Code below is intended to be reproducible and submission-focused.


## Reference Executed Outputs From Good Runs

These are copied from the successful source notebooks and kept as markdown reference only. They are not manual leaderboard notes inside code.

### Pseudo-GT build: `labels++.ipynb`, Codex pseudo-GT

```text
Total tiles: 16
Mean pos_frac: 14.529%
Mean neg_frac: 81.841%
Mean ign_frac: 3.630%
Total strong pos: 944,907
Total medium pos: 650,979
Total weak pos: 728,679

Positives per year:
  2020: 479,827
  2021: 504,111
  2022: 396,607
  2023: 430,751
  2024: 371,878
  2025: 141,391
```

### LightGBM all-year raw AEF run: `labels++.ipynb`

```text
=== Step 1: Extract features (target = 389 per pixel) ===
Total labeled pixels: 15,419,204
Positives: 2,324,565 (15.08%)
Negatives: 13,094,639
Feature matrix: (15419204, 389), dtype=float32
Memory: 23.99 GB

=== Step 2: Subsample negatives 5x ===
After subsampling: 13,947,390 rows (16.7% positive)
Memory: 21.70 GB

Validation tile: 18NXH_6_8
  Train rows: 13,133,371, Val rows: 814,019

=== Step 4: Train LightGBM ===
[50]  train binary_logloss: 0.116353  val binary_logloss: 0.101370
[100] train binary_logloss: 0.094752  val binary_logloss: 0.069165
[150] train binary_logloss: 0.087677  val binary_logloss: 0.062654
[200] train binary_logloss: 0.083303  val binary_logloss: 0.059736
[250] train binary_logloss: 0.080151  val binary_logloss: 0.057794
[300] train binary_logloss: 0.077673  val binary_logloss: 0.056163
[350] train binary_logloss: 0.075630  val binary_logloss: 0.054976
[400] train binary_logloss: 0.073927  val binary_logloss: 0.054745
Best iteration: 400

=== Step 5: Tune threshold ===
Best threshold: 0.25 (F1=0.968)

=== Step 6: Predict test tiles ===
18NVJ_1_6: 3,193 positive pixels, 8 polygons
18NYH_2_1: 109,588 positive pixels, 202 polygons
33NTE_5_1: 65,544 positive pixels, 223 polygons
47QMA_6_2: 18,718 positive pixels, 97 polygons
48PWA_0_6: 154,803 positive pixels, 312 polygons

Submission: submission/submission.geojson
Total polygons: 842
```

### LightGBM saved all-tile model at threshold 0.15: `labels++.ipynb`

```text
Loading model: eda_artifacts/model/lgbm_allyears_alltiles_from_memory.txt
Found 5 test tiles
Using threshold: 0.15
TARGET_SHAPE: (1000, 1000)

18NVJ_1_6 thr015: 10,556 positive pixels, 42 polygons
18NYH_2_1 thr015: 131,476 positive pixels, 215 polygons
33NTE_5_1 thr015: 83,873 positive pixels, 247 polygons
47QMA_6_2 thr015: 0 positive pixels, skipped geojson
48PWA_0_6 thr015: 193,186 positive pixels, 310 polygons

Saved submission: eda_artifacts/submissions/submission_allyears_alltiles_thr015.geojson
Total polygons: 814
Empty tiles skipped: 1
```

### LightGBM top feature importance: `labels++.ipynb`

```text
feature              gain
change_2022_2023    1.009768e+07
aef_2020_22         6.763814e+06
change_2023_2024    5.492729e+06
aef_2020_44         3.707022e+06
change_2020_2021    3.601727e+06
aef_2025_5          3.102407e+06
change_2024_2025    2.716207e+06
aef_2020_5          2.015630e+06
change_2021_2022    1.864243e+06
aef_2020_0          1.491067e+06
```

### XGBoost all-year raw AEF run: `xgboost.ipynb`

```text
Feature matrix: (15419204, 389), dtype=float32
After subsampling: 13,947,390 rows (16.7% positive)
Validation tile: 18NXH_6_8
  Train rows: 13,133,371, Val rows: 814,019

=== Step 4: Train XGBoost ===
[0]   train-logloss:0.36555  val-logloss:0.58167
[100] train-logloss:0.09972  val-logloss:0.06188
[200] train-logloss:0.08906  val-logloss:0.05295
[300] train-logloss:0.08417  val-logloss:0.05056
[400] train-logloss:0.08068  val-logloss:0.04906
[500] train-logloss:0.07807  val-logloss:0.04775
[600] train-logloss:0.07605  val-logloss:0.04676
[700] train-logloss:0.07447  val-logloss:0.04592
[800] train-logloss:0.07301  val-logloss:0.04535
[900] train-logloss:0.07168  val-logloss:0.04487
[999] train-logloss:0.07042  val-logloss:0.04444

Saved model: eda_artifacts/model/xgb_allyears.json
Fixed threshold: 0.15
Validation F1 at fixed threshold: 0.966
```

### XGBoost saved model submission at threshold 0.15: `xgboost.ipynb`

```text
Loaded model: eda_artifacts/model/xgb_allyears.json
18NVJ_1_6: 3,361 positive pixels, 13 polygons
18NYH_2_1: 124,373 positive pixels, 200 polygons
33NTE_5_1: 87,232 positive pixels, 251 polygons
47QMA_6_2: 24,284 positive pixels, 130 polygons
48PWA_0_6: 189,084 positive pixels, 327 polygons

Saved submission: submission/submission_xgb_allyears_saved_thr015.geojson
Total polygons: 921
```


## 1. Imports + Config


In [ ]:
from pathlib import Path
from datetime import date
import json
import re
import sys
import warnings

import numpy as np
import pandas as pd
import rasterio
from rasterio.features import shapes as raster_shapes
from rasterio.warp import reproject, Resampling
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm.auto import tqdm

try:
    from scipy.ndimage import label as cc_label
except Exception as exc:
    cc_label = None
    print(f"scipy.ndimage.label unavailable: {exc}")

sys.path.insert(0, ".")
try:
    from submission_utils import raster_to_geojson
except Exception as exc:
    print(f"Using local raster_to_geojson fallback: {exc}")

    def raster_to_geojson(raster_path, output_path=None):
        features = []
        with rasterio.open(raster_path) as src:
            arr = src.read(1)
            mask = arr > 0
            for geom, value in raster_shapes(arr.astype(np.uint8), mask=mask, transform=src.transform):
                if int(value) == 0:
                    continue
                features.append({
                    "type": "Feature",
                    "properties": {"value": int(value)},
                    "geometry": geom,
                })

        out = {"type": "FeatureCollection", "features": features}
        if output_path is not None:
            with open(output_path, "w") as f:
                json.dump(out, f)
        return out


ROOT = Path("data/makeathon-challenge")
PGT_DIR = Path("eda_artifacts/pseudo_gt_codex")
CLOUD_ROOT = ROOT / "processed/s2_cloudmask"

MODEL_DIR = Path("eda_artifacts/model")
PRED_DIR = Path("eda_artifacts/predictions")
SUB_DIR = Path("submission")
META_DIR = Path("eda_artifacts/run_metadata")

for d in [PGT_DIR, MODEL_DIR, PRED_DIR, SUB_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)


YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
TARGET_SHAPE = (1000, 1000)
N_AEF_BANDS = 64
N_FEATURES = len(YEARS) * N_AEF_BANDS + (len(YEARS) - 1)

CV_HOLDOUT = "18NXH_6_8"
NEG_RATIO = 5
RANDOM_SEED = 42

THRESHOLDS = [0.15, 0.16, 0.17, 0.18, 0.20]

BUILD_PSEUDO_GT_IF_MISSING = True
RUN_VALIDATION = True
RUN_FINAL_ALL_TILES = True
RUN_LGB = True
RUN_XGB = True
RUN_TEST_PREDICTIONS = True
RUN_THRESHOLD_SUBMISSIONS = True
RUN_ENSEMBLE_SUBMISSIONS = True

LGB_NUM_BOOST_ROUND = 400
XGB_NUM_BOOST_ROUND = 800

LGB_FINAL_TAG = "lgb_alltiles"
XGB_ENSEMBLE_TAG = "xgb_holdout_best"

warnings.filterwarnings("ignore", category=RuntimeWarning)
print(f"N_FEATURES={N_FEATURES}")


## 2. Pseudo-GT Builder

This is the Codex pseudo-GT path that produced the current training labels. It runs only when labels are missing or when called explicitly.


In [ ]:
DATE_AGREEMENT_DAYS = 90
WINDOW_START = date(2020, 1, 1).toordinal()
WINDOW_END = date(2026, 1, 1).toordinal()

EPOCH_RADD = date(2014, 12, 31).toordinal()
EPOCH_GLADS2 = date(2019, 1, 1).toordinal()

RADD_WEIGHT_CLEAR = 0.55
RADD_WEIGHT_CLOUDY = 0.80
OPTICAL_WEIGHT_CLEAR = 0.35
OPTICAL_WEIGHT_CLOUDY = 0.10
RADAR_OPTICAL_PAIR_BONUS = 0.10
OPTICAL_PAIR_BONUS_CLEAR = 0.15
OPTICAL_PAIR_BONUS_CLOUDY = 0.02
POSITIVE_CONF_THRESHOLD = 0.50
NEGATIVE_CONFIDENCE = 1.00


def get_train_label_ref(tile):
    s2_dir = ROOT / f"sentinel-2/train/{tile}__s2_l2a"
    if s2_dir.exists():
        best = None
        best_area = 0
        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as src:
                h, w = src.shape
                area = h * w
                if h >= 300 and w >= 300 and area > best_area:
                    best_area = area
                    best = (src.transform, src.crs, src.shape)
        if best is not None:
            return best

    gl_files = sorted((ROOT / "labels/train/gladl").glob(f"gladl_{tile}_alert*.tif"))
    if gl_files:
        with rasterio.open(gl_files[0]) as src:
            return src.transform, src.crs, src.shape

    radd_path = ROOT / f"labels/train/radd/radd_{tile}_labels.tif"
    if radd_path.exists():
        with rasterio.open(radd_path) as src:
            return src.transform, src.crs, src.shape

    return None


def reproj_label(path, ref, dtype):
    out = np.zeros(ref[2], dtype=dtype)
    with rasterio.open(path) as src:
        reproject(
            rasterio.band(src, 1),
            out,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref[0],
            dst_crs=ref[1],
            resampling=Resampling.nearest,
        )
    return out


def resize_label_nearest(arr, target_shape):
    h_src, w_src = arr.shape
    h_tgt, w_tgt = target_shape
    rows = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    cols = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)
    return arr[rows[:, None], cols[None, :]]


def ordinal_to_year_month(ord_arr):
    year = np.zeros(ord_arr.shape, dtype=np.int16)
    month = np.zeros(ord_arr.shape, dtype=np.int8)

    for ord_val in np.unique(ord_arr[ord_arr > 0]):
        d = date.fromordinal(int(ord_val))
        sel = ord_arr == ord_val
        year[sel] = d.year
        month[sel] = d.month

    return year, month


def find_cloud_mask_path(tile, year, month):
    tile_dir = CLOUD_ROOT / "train" / f"{tile}__s2_l2a"
    if not tile_dir.exists():
        return None

    candidates = [
        tile_dir / f"{tile}__s2_l2a_{year}_{month}_cloudmask.tif",
        tile_dir / f"{tile}__s2_l2a_{year}_{month:02d}_cloudmask.tif",
    ]

    for p in candidates:
        if p.exists():
            return p

    for pattern in [
        f"{tile}__s2_l2a_{year}_{month}*_cloudmask.tif",
        f"{tile}__s2_l2a_{year}_{month:02d}*_cloudmask.tif",
    ]:
        fuzzy = sorted(tile_dir.glob(pattern))
        if fuzzy:
            return fuzzy[0]

    return None


def load_cloud_mask_on_ref(tile, year, month, ref):
    p = find_cloud_mask_path(tile, year, month)
    if p is None:
        return None

    arr = reproj_label(p, ref, dtype=np.uint8)
    return arr == 1


def cloudy_from_dates(tile, date_ord, ref):
    cloudy_any = np.zeros(ref[2], dtype=bool)
    year_arr, month_arr = ordinal_to_year_month(date_ord)
    pairs = np.unique(np.stack([year_arr, month_arr], axis=-1).reshape(-1, 2), axis=0)

    for y, m in pairs:
        y = int(y)
        m = int(m)
        if y <= 0 or m <= 0:
            continue

        cloudy = load_cloud_mask_on_ref(tile, y, m, ref)
        if cloudy is None:
            continue

        sel = (year_arr == y) & (month_arr == m)
        cloudy_any[sel] = cloudy[sel]

    return cloudy_any


def get_radd(tile, ref):
    p = ROOT / f"labels/train/radd/radd_{tile}_labels.tif"
    if not p.exists():
        return None, None, False

    raw = reproj_label(p, ref, dtype=np.int32)
    days = raw % 10000
    date_ord = np.where(raw > 0, days + EPOCH_RADD, 0)
    in_window = (date_ord >= WINDOW_START) & (date_ord < WINDOW_END)
    mask = (raw > 0) & in_window

    return mask, np.where(mask, date_ord, 0), True


def get_glads2(tile, ref):
    alert_path = ROOT / f"labels/train/glads2/glads2_{tile}_alert.tif"
    date_path = ROOT / f"labels/train/glads2/glads2_{tile}_alertDate.tif"

    if not alert_path.exists() or not date_path.exists():
        return None, None, False

    alert = reproj_label(alert_path, ref, dtype=np.uint8)
    days = reproj_label(date_path, ref, dtype=np.uint16)
    date_ord = np.where(alert > 0, days.astype(np.int64) + EPOCH_GLADS2, 0)
    in_window = (date_ord >= WINDOW_START) & (date_ord < WINDOW_END)
    mask = (alert >= 2) & in_window

    return mask, np.where(mask, date_ord, 0), True


def get_gladl(tile, ref):
    any_file = False
    mask = np.zeros(ref[2], dtype=bool)
    date_ord = np.zeros(ref[2], dtype=np.int64)

    for year in YEARS:
        yy = year % 100
        alert_path = ROOT / f"labels/train/gladl/gladl_{tile}_alert{yy:02d}.tif"
        date_path = ROOT / f"labels/train/gladl/gladl_{tile}_alertDate{yy:02d}.tif"

        if not alert_path.exists() or not date_path.exists():
            continue

        any_file = True
        alert = reproj_label(alert_path, ref, dtype=np.uint8)
        doy = reproj_label(date_path, ref, dtype=np.uint16)

        year_start = date(year, 1, 1).toordinal()
        this_date = np.where(alert > 0, doy.astype(np.int64) + year_start - 1, 0)
        this_mask = alert > 0
        better = this_mask & ((date_ord == 0) | ((this_date > 0) & (this_date < date_ord)))

        date_ord = np.where(better, this_date, date_ord)
        mask |= this_mask

    if not any_file:
        return None, None, False

    return mask, date_ord, True


def close_date_mask(m1, d1, m2, d2):
    both = m1 & m2
    if not both.any():
        return np.zeros(m1.shape, dtype=bool)

    diff = np.abs(d1.astype(np.int64) - d2.astype(np.int64))
    return both & (diff <= DATE_AGREEMENT_DAYS) & (d1 > 0) & (d2 > 0)


def pair_bonus(m1, d1, m2, d2, value):
    close = close_date_mask(m1, d1, m2, d2)
    return np.where(close, value, 0.0).astype(np.float32)


def optical_pair_bonus(tile, gs2_mask, gs2_date, gl_mask, gl_date, ref):
    close = close_date_mask(gs2_mask, gs2_date, gl_mask, gl_date)
    if not close.any():
        return np.zeros(ref[2], dtype=np.float32)

    gs2_cloudy = cloudy_from_dates(tile, gs2_date, ref)
    gl_cloudy = cloudy_from_dates(tile, gl_date, ref)
    cloudy_pair = gs2_cloudy | gl_cloudy

    bonus = np.where(cloudy_pair, OPTICAL_PAIR_BONUS_CLOUDY, OPTICAL_PAIR_BONUS_CLEAR)
    return np.where(close, bonus, 0.0).astype(np.float32)


def build_confidence(tile, ref, radd_mask, radd_date, gs2_mask, gs2_date, gl_mask, gl_date):
    radd_cloudy = cloudy_from_dates(tile, radd_date, ref)
    gs2_cloudy = cloudy_from_dates(tile, gs2_date, ref)
    gl_cloudy = cloudy_from_dates(tile, gl_date, ref)

    radd_weight = np.where(radd_cloudy, RADD_WEIGHT_CLOUDY, RADD_WEIGHT_CLEAR).astype(np.float32)
    gs2_weight = np.where(gs2_cloudy, OPTICAL_WEIGHT_CLOUDY, OPTICAL_WEIGHT_CLEAR).astype(np.float32)
    gl_weight = np.where(gl_cloudy, OPTICAL_WEIGHT_CLOUDY, OPTICAL_WEIGHT_CLEAR).astype(np.float32)

    confidence = np.zeros(ref[2], dtype=np.float32)
    confidence += np.where(radd_mask, radd_weight, 0.0).astype(np.float32)
    confidence += np.where(gs2_mask, gs2_weight, 0.0).astype(np.float32)
    confidence += np.where(gl_mask, gl_weight, 0.0).astype(np.float32)
    confidence += pair_bonus(radd_mask, radd_date, gs2_mask, gs2_date, RADAR_OPTICAL_PAIR_BONUS)
    confidence += pair_bonus(radd_mask, radd_date, gl_mask, gl_date, RADAR_OPTICAL_PAIR_BONUS)
    confidence += optical_pair_bonus(tile, gs2_mask, gs2_date, gl_mask, gl_date, ref)

    return np.clip(confidence, 0.0, 1.0)


def build_pseudo_gt(tile):
    ref = get_train_label_ref(tile)
    if ref is None:
        return None

    shape = ref[2]
    radd_mask, radd_date, has_radd = get_radd(tile, ref)
    gs2_mask, gs2_date, has_gs2 = get_glads2(tile, ref)
    gl_mask, gl_date, has_gl = get_gladl(tile, ref)

    if radd_mask is None:
        radd_mask = np.zeros(shape, dtype=bool)
        radd_date = np.zeros(shape, dtype=np.int64)

    if gs2_mask is None:
        gs2_mask = np.zeros(shape, dtype=bool)
        gs2_date = np.zeros(shape, dtype=np.int64)

    if gl_mask is None:
        gl_mask = np.zeros(shape, dtype=bool)
        gl_date = np.zeros(shape, dtype=np.int64)

    votes = radd_mask.astype(np.int8) + gs2_mask.astype(np.int8) + gl_mask.astype(np.int8)
    n_sources = int(has_radd) + int(has_gs2) + int(has_gl)

    confidence = build_confidence(tile, ref, radd_mask, radd_date, gs2_mask, gs2_date, gl_mask, gl_date)
    label = np.full(shape, np.nan, dtype=np.float32)
    label[confidence >= POSITIVE_CONF_THRESHOLD] = 1.0

    if n_sources > 0:
        label[votes == 0] = 0.0

    label[(votes > 0) & (confidence < POSITIVE_CONF_THRESHOLD)] = np.nan
    confidence = np.where(np.isnan(label), 0.0, confidence)
    confidence = np.where(label == 0.0, NEGATIVE_CONFIDENCE, confidence)

    event_month = np.zeros(shape, dtype=np.int8)
    event_year = np.zeros(shape, dtype=np.int16)

    dates = np.stack([radd_date, gs2_date, gl_date])
    masked_dates = np.where(dates > 0, dates, np.iinfo(np.int64).max)
    earliest = masked_dates.min(axis=0)
    has_date = earliest < np.iinfo(np.int64).max

    for ord_val in np.unique(earliest[has_date]):
        sel = (earliest == ord_val) & has_date & (label == 1.0)
        if not sel.any():
            continue
        d = date.fromordinal(int(ord_val))
        event_month[sel] = d.month
        event_year[sel] = d.year

    return {
        "label": resize_label_nearest(label, TARGET_SHAPE),
        "confidence": resize_label_nearest(confidence, TARGET_SHAPE),
        "event_month": resize_label_nearest(event_month, TARGET_SHAPE),
        "event_year": resize_label_nearest(event_year, TARGET_SHAPE),
        "native_shape": f"{shape[0]}x{shape[1]}",
        "n_sources": n_sources,
        "has_radd": has_radd,
        "has_gs2": has_gs2,
        "has_gl": has_gl,
    }


In [ ]:
def list_train_tiles():
    train_root = ROOT / "sentinel-2/train"
    if not train_root.exists():
        raise FileNotFoundError(f"Missing train directory: {train_root}")

    return sorted(
        p.name.replace("__s2_l2a", "")
        for p in train_root.iterdir()
        if p.is_dir() and p.name.endswith("__s2_l2a")
    )


def build_all_pseudo_gt(overwrite=False):
    train_tiles = list_train_tiles()
    stats = []

    for tile in tqdm(train_tiles, desc="pseudo GT"):
        out_path = PGT_DIR / f"{tile}.npz"
        if out_path.exists() and not overwrite:
            continue

        result = build_pseudo_gt(tile)
        if result is None:
            print(f"{tile}: skipped")
            continue

        np.savez_compressed(
            out_path,
            label=result["label"],
            confidence=result["confidence"],
            event_month=result["event_month"],
            event_year=result["event_year"],
        )

        label = result["label"]
        confidence = result["confidence"]
        event_year = result["event_year"]

        pos = label == 1.0
        neg = label == 0.0
        ign = np.isnan(label)

        row = {
            "tile": tile,
            "native_shape": result["native_shape"],
            "n_sources": result["n_sources"],
            "has_radd": result["has_radd"],
            "has_gs2": result["has_gs2"],
            "has_gl": result["has_gl"],
            "pos_frac": float(pos.mean()),
            "neg_frac": float(neg.mean()),
            "ign_frac": float(ign.mean()),
            "pos_mean_conf": float(confidence[pos].mean()) if pos.any() else 0.0,
            "strong_pos": int((pos & (confidence >= 0.95)).sum()),
            "medium_pos": int((pos & (confidence >= 0.80) & (confidence < 0.95)).sum()),
            "weak_pos": int((pos & (confidence < 0.80)).sum()),
        }

        for year in YEARS:
            row[f"pos_y{year}"] = int((pos & (event_year == year)).sum())

        stats.append(row)

    if stats:
        df = pd.DataFrame(stats)
        df.to_csv(PGT_DIR / "stats.csv", index=False)
        display(df)
    else:
        print("No pseudo-GT files were built. Existing files will be reused if present.")


if BUILD_PSEUDO_GT_IF_MISSING:
    train_root = ROOT / "sentinel-2/train"
    if train_root.exists():
        missing = [tile for tile in list_train_tiles() if not (PGT_DIR / f"{tile}.npz").exists()]
        print(f"Missing pseudo-GT files: {len(missing)}")
        if missing:
            build_all_pseudo_gt(overwrite=False)
    else:
        print(f"Data directory not available yet: {train_root}")


## 3. Utility Helpers


In [ ]:
def get_ref(tile, split):
    s2_dir = ROOT / f"sentinel-2/{split}/{tile}__s2_l2a"

    if s2_dir.exists():
        best = None
        best_area = 0

        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                area = h * w

                if h >= 300 and w >= 300 and area > best_area:
                    best_area = area
                    best = (r.transform, r.crs, r.shape)

        if best is not None:
            return best

    aef_files = sorted((ROOT / f"aef-embeddings/{split}").glob(f"{tile}_*.tiff"))

    if aef_files:
        with rasterio.open(aef_files[0]) as r:
            return r.transform, r.crs, r.shape

    return None


def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape[-2:]
    h_tgt, w_tgt = target_shape

    row_idx = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    col_idx = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)

    if arr.ndim == 2:
        return arr[row_idx[:, None], col_idx[None, :]]

    return arr[:, row_idx[:, None], col_idx[None, :]]


def native_from_target(mask_or_prob, native_shape):
    h_native, w_native = native_shape
    h_tgt, w_tgt = TARGET_SHAPE

    row_idx = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
    col_idx = (np.arange(w_native) * w_tgt // w_native).clip(0, w_tgt - 1)

    return mask_or_prob[row_idx[:, None], col_idx[None, :]]


def feature_names():
    names = []

    for y in YEARS:
        for b in range(N_AEF_BANDS):
            names.append(f"aef_{y}_{b}")

    for i in range(len(YEARS) - 1):
        names.append(f"change_{YEARS[i]}_{YEARS[i + 1]}")

    return names


def list_split_tiles(split):
    split_root = ROOT / f"sentinel-2/{split}"
    if not split_root.exists():
        raise FileNotFoundError(f"Missing split directory: {split_root}")

    return sorted(
        p.name.replace("__s2_l2a", "")
        for p in split_root.iterdir()
        if p.is_dir() and p.name.endswith("__s2_l2a")
    )


def write_binary_tif(binary_native, ref, tile_path):
    h_native, w_native = ref[2]
    profile = {
        "driver": "GTiff",
        "height": h_native,
        "width": w_native,
        "count": 1,
        "dtype": "uint8",
        "crs": ref[1],
        "transform": ref[0],
        "nodata": 0,
    }

    with rasterio.open(tile_path, "w", **profile) as dst:
        dst.write(binary_native.astype(np.uint8), 1)


## 4. AEF Feature Extraction


In [ ]:
def load_aef_to_ref(tile, year, split, ref):
    p = ROOT / f"aef-embeddings/{split}/{tile}_{year}.tiff"

    if not p.exists():
        return None

    aef = np.zeros((N_AEF_BANDS, *ref[2]), dtype=np.float32)

    with rasterio.open(p) as src:
        raw = src.read().astype(np.float32)
        raw = np.where(np.isfinite(raw), raw, 0.0)

        for b in range(N_AEF_BANDS):
            reproject(
                raw[b],
                aef[b],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref[0],
                dst_crs=ref[1],
                resampling=Resampling.bilinear,
            )

    return aef


def extract_features(tile, split):
    ref = get_ref(tile, split)

    if ref is None:
        return None, None

    aef_stack = []

    for year in YEARS:
        aef = load_aef_to_ref(tile, year, split, ref)

        if aef is None:
            print(f"Missing AEF for {tile} year {year}")
            return None, None

        aef = resize_nearest(aef, TARGET_SHAPE)
        aef_stack.append(aef)

    changes = []

    for i in range(len(YEARS) - 1):
        a = aef_stack[i]
        b = aef_stack[i + 1]

        dot = (a * b).sum(axis=0)
        norm_a = np.linalg.norm(a, axis=0)
        norm_b = np.linalg.norm(b, axis=0)
        denom = norm_a * norm_b

        cos_sim = np.where(
            denom > 1e-6,
            dot / (denom + 1e-9),
            0.0,
        ).astype(np.float32)

        changes.append(cos_sim)

    all_feats = np.concatenate(
        aef_stack + [c[None] for c in changes],
        axis=0,
    )

    features = all_feats.reshape(N_FEATURES, -1).T.astype(np.float32)
    return features, ref


## 5. Build Training Matrix


In [ ]:
def build_training_matrix():
    train_tiles = list_split_tiles("train")

    all_feats = []
    all_labels = []
    all_weights = []
    all_tiles = []

    for tile in tqdm(train_tiles, desc="train features"):
        pgt_path = PGT_DIR / f"{tile}.npz"

        if not pgt_path.exists():
            print(f"{tile}: missing pseudo-GT")
            continue

        feats, _ = extract_features(tile, "train")

        if feats is None:
            continue

        pgt = np.load(pgt_path)
        label = pgt["label"].flatten().astype(np.float32)
        conf = pgt["confidence"].flatten().astype(np.float32)

        valid = ~np.isnan(label)

        all_feats.append(feats[valid])
        all_labels.append(label[valid])
        all_weights.append(conf[valid])
        all_tiles.append(np.full(int(valid.sum()), tile))

    if not all_feats:
        raise RuntimeError("No training data was built. Check ROOT and PGT_DIR.")

    X = np.concatenate(all_feats, axis=0).astype(np.float32)
    y = np.concatenate(all_labels).astype(np.float32)
    w = np.concatenate(all_weights).astype(np.float32)
    tile_ids = np.concatenate(all_tiles)

    return X, y, w, tile_ids


def subsample_negatives(X, y, w, tile_ids, neg_ratio=5, seed=42):
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    n_neg_keep = min(len(neg_idx), len(pos_idx) * neg_ratio)

    rng = np.random.default_rng(seed)
    neg_sampled = rng.choice(neg_idx, size=n_neg_keep, replace=False)

    keep = np.concatenate([pos_idx, neg_sampled])
    rng.shuffle(keep)

    return X[keep], y[keep], w[keep], tile_ids[keep]


In [ ]:
X, y, w, tile_ids = build_training_matrix()

print(f"Full matrix: {X.shape}, memory={X.nbytes / 1e9:.2f} GB")
print(f"Positive rate: {(y == 1).mean():.2%}")

X_kept, y_kept, w_kept, tiles_kept = subsample_negatives(
    X, y, w, tile_ids,
    neg_ratio=NEG_RATIO,
    seed=RANDOM_SEED,
)

print(f"Sampled matrix: {X_kept.shape}, memory={X_kept.nbytes / 1e9:.2f} GB")
print(f"Sampled positive rate: {(y_kept == 1).mean():.2%}")
print(f"Tiles: {sorted(set(tiles_kept.tolist()))}")


## 6. Train LightGBM


In [ ]:
names = feature_names()

lgb_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
    "seed": RANDOM_SEED,
}

lgb_val_model = None
lgb_final_model = None

if RUN_LGB and RUN_VALIDATION:
    val_mask = tiles_kept == CV_HOLDOUT
    tr_mask = ~val_mask

    lgb_train = lgb.Dataset(
        X_kept[tr_mask],
        y_kept[tr_mask],
        weight=w_kept[tr_mask],
        feature_name=names,
    )

    lgb_val = lgb.Dataset(
        X_kept[val_mask],
        y_kept[val_mask],
        weight=w_kept[val_mask],
        reference=lgb_train,
    )

    lgb_val_model = lgb.train(
        lgb_params,
        lgb_train,
        num_boost_round=LGB_NUM_BOOST_ROUND,
        valid_sets=[lgb_train, lgb_val],
        valid_names=["train", "val"],
        callbacks=[
            lgb.early_stopping(30),
            lgb.log_evaluation(50),
        ],
    )

    lgb_val_model.save_model(str(MODEL_DIR / "lgb_allyears_holdout.txt"))

    val_pred = lgb_val_model.predict(X_kept[val_mask])

    rows = []
    for thr in THRESHOLDS:
        yp = val_pred > thr
        rows.append({
            "threshold": thr,
            "f1": f1_score(y_kept[val_mask], yp),
            "precision": precision_score(y_kept[val_mask], yp, zero_division=0),
            "recall": recall_score(y_kept[val_mask], yp, zero_division=0),
            "pred_frac": float(yp.mean()),
            "true_frac": float((y_kept[val_mask] == 1).mean()),
        })

    display(pd.DataFrame(rows))

if RUN_LGB and RUN_FINAL_ALL_TILES:
    lgb_all = lgb.Dataset(
        X_kept,
        y_kept,
        weight=w_kept,
        feature_name=names,
    )

    lgb_final_model = lgb.train(
        lgb_params,
        lgb_all,
        num_boost_round=LGB_NUM_BOOST_ROUND,
        callbacks=[lgb.log_evaluation(50)],
    )

    lgb_final_model.save_model(str(MODEL_DIR / "lgb_allyears_alltiles.txt"))


## 7. Train XGBoost


In [ ]:
xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "eta": 0.05,
    "max_depth": 6,
    "min_child_weight": 200,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "lambda": 1.0,
    "alpha": 0.0,
    "tree_method": "hist",
    "max_bin": 256,
    "nthread": -1,
    "seed": RANDOM_SEED,
}

xgb_val_model = None
xgb_final_model = None

if RUN_XGB and RUN_VALIDATION:
    val_mask = tiles_kept == CV_HOLDOUT
    tr_mask = ~val_mask

    dtrain = xgb.DMatrix(
        X_kept[tr_mask],
        label=y_kept[tr_mask],
        weight=w_kept[tr_mask],
        feature_names=names,
    )

    dval = xgb.DMatrix(
        X_kept[val_mask],
        label=y_kept[val_mask],
        weight=w_kept[val_mask],
        feature_names=names,
    )

    xgb_val_model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=XGB_NUM_BOOST_ROUND,
        evals=[(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=50,
        verbose_eval=50,
    )

    xgb_val_model.save_model(str(MODEL_DIR / "xgb_allyears_holdout.json"))

if RUN_XGB and RUN_FINAL_ALL_TILES:
    dall = xgb.DMatrix(
        X_kept,
        label=y_kept,
        weight=w_kept,
        feature_names=names,
    )

    xgb_final_model = xgb.train(
        xgb_params,
        dall,
        num_boost_round=XGB_NUM_BOOST_ROUND,
        verbose_eval=50,
    )

    xgb_final_model.save_model(str(MODEL_DIR / "xgb_allyears_alltiles.json"))


## 8. Predict Test Probabilities


In [ ]:
def load_lgb_model_if_needed(model, path):
    if model is not None:
        return model
    if Path(path).exists():
        return lgb.Booster(model_file=str(path))
    return None


def load_xgb_model_if_needed(model, path):
    if model is not None:
        return model
    if Path(path).exists():
        booster = xgb.Booster()
        booster.load_model(str(path))
        return booster
    return None


def predict_test_lgb(model, tag):
    test_tiles = list_split_tiles("test")

    for tile in tqdm(test_tiles, desc=f"predict {tag}"):
        feats, _ = extract_features(tile, "test")

        if feats is None:
            print(f"{tile}: skipped")
            continue

        prob = model.predict(feats).astype(np.float32).reshape(TARGET_SHAPE)
        np.save(PRED_DIR / f"{tile}_prob_{tag}.npy", prob)

    print(f"Saved test probabilities for {tag}")


def predict_test_xgb(model, tag, iteration_range=None):
    test_tiles = list_split_tiles("test")

    for tile in tqdm(test_tiles, desc=f"predict {tag}"):
        feats, _ = extract_features(tile, "test")

        if feats is None:
            print(f"{tile}: skipped")
            continue

        dtest = xgb.DMatrix(feats, feature_names=names)

        if iteration_range is None:
            pred = model.predict(dtest)
        else:
            pred = model.predict(dtest, iteration_range=iteration_range)

        prob = pred.astype(np.float32).reshape(TARGET_SHAPE)
        np.save(PRED_DIR / f"{tile}_prob_{tag}.npy", prob)

    print(f"Saved test probabilities for {tag}")


In [ ]:
if RUN_TEST_PREDICTIONS:
    lgb_final_model = load_lgb_model_if_needed(lgb_final_model, MODEL_DIR / "lgb_allyears_alltiles.txt")
    if lgb_final_model is not None:
        predict_test_lgb(lgb_final_model, LGB_FINAL_TAG)
    else:
        print("No LightGBM final model available for prediction.")

    if xgb_val_model is not None:
        best_iter = xgb_val_model.best_iteration + 1
        predict_test_xgb(xgb_val_model, XGB_ENSEMBLE_TAG, iteration_range=(0, best_iter))
    else:
        xgb_saved = load_xgb_model_if_needed(None, MODEL_DIR / "xgb_allyears_holdout.json")
        if xgb_saved is not None:
            predict_test_xgb(xgb_saved, XGB_ENSEMBLE_TAG)
        else:
            print("No XGBoost holdout model available for prediction.")


## 9. Make Submissions From Probabilities


In [ ]:
def make_submission_from_probs(prob_tag, threshold, out_name):
    test_tiles = list_split_tiles("test")
    all_features = []

    for tile in test_tiles:
        prob_path = PRED_DIR / f"{tile}_prob_{prob_tag}.npy"

        if not prob_path.exists():
            print(f"{tile}: missing {prob_path}")
            continue

        ref = get_ref(tile, "test")
        prob = np.load(prob_path).astype(np.float32)
        binary = (prob > threshold).astype(np.uint8)
        binary_native = native_from_target(binary, ref[2])

        tile_path = PRED_DIR / f"{tile}_{out_name}.tif"
        write_binary_tif(binary_native, ref, tile_path)

        if int(binary_native.sum()) == 0:
            print(f"{tile}: px=0, polygons=0")
            continue

        geojson = raster_to_geojson(str(tile_path), output_path=None)

        for feat in geojson["features"]:
            feat.setdefault("properties", {})["tile"] = tile

        all_features.extend(geojson["features"])

        print(f"{tile}: px={int(binary_native.sum()):,}, polygons={len(geojson['features'])}")

    submission = {
        "type": "FeatureCollection",
        "features": all_features,
    }

    out_path = SUB_DIR / f"{out_name}.geojson"

    with open(out_path, "w") as f:
        json.dump(submission, f)

    print(f"Saved: {out_path}")
    print(f"Total polygons: {len(all_features)}")
    return out_path


In [ ]:
if RUN_THRESHOLD_SUBMISSIONS:
    for thr in THRESHOLDS:
        make_submission_from_probs(
            prob_tag=LGB_FINAL_TAG,
            threshold=thr,
            out_name=f"submission_lgb_alltiles_thr{int(thr * 100):03d}",
        )


## 10. Ensemble Submissions

Primary ensemble framing:

```text
LGB > 0.25 OR (XGB > 0.17 AND LGB > 0.10)
```

This is ensemble learning over two tabular models trained on the same all-year raw AEF feature space. The optional weak-component filter is post-processing on the ensemble mask, not a separate experiment branch.


In [ ]:
def remove_weak_tiny_components(mask, score, min_area=25, min_mean_score=0.20):
    if cc_label is None:
        raise ImportError("scipy is required for connected-component filtering")

    lab, n = cc_label(mask)

    if n == 0:
        return mask

    keep = np.zeros(n + 1, dtype=bool)

    for i in range(1, n + 1):
        comp = lab == i
        area = int(comp.sum())
        mean_score = float(score[comp].mean()) if area > 0 else 0.0

        if area >= min_area or mean_score >= min_mean_score:
            keep[i] = True

    return keep[lab]


def make_ensemble_submission(tag_a, tag_b, out_name, mode="hybrid", threshold=0.17):
    test_tiles = list_split_tiles("test")
    all_features = []

    for tile in test_tiles:
        pa = PRED_DIR / f"{tile}_prob_{tag_a}.npy"
        pb = PRED_DIR / f"{tile}_prob_{tag_b}.npy"

        if not pa.exists() or not pb.exists():
            print(f"{tile}: missing probs")
            continue

        prob_a = np.load(pa).astype(np.float32)
        prob_b = np.load(pb).astype(np.float32)

        if mode == "blend":
            score = 0.65 * prob_a + 0.35 * prob_b
            mask = score > threshold

        elif mode == "hybrid":
            score = np.maximum(prob_a, prob_b)
            mask = (prob_b > 0.25) | ((prob_a > 0.17) & (prob_b > 0.10))

        elif mode == "hybrid_filter":
            score = np.maximum(prob_a, prob_b)
            mask = (prob_b > 0.25) | ((prob_a > 0.17) & (prob_b > 0.10))
            mask = remove_weak_tiny_components(
                mask,
                score,
                min_area=25,
                min_mean_score=0.20,
            )

        else:
            raise ValueError(mode)

        ref = get_ref(tile, "test")
        binary_native = native_from_target(mask.astype(np.uint8), ref[2])
        tile_path = PRED_DIR / f"{tile}_{out_name}.tif"
        write_binary_tif(binary_native, ref, tile_path)

        if int(binary_native.sum()) == 0:
            print(f"{tile}: px=0, polygons=0")
            continue

        geojson = raster_to_geojson(str(tile_path), output_path=None)

        for feat in geojson["features"]:
            feat.setdefault("properties", {})["tile"] = tile

        all_features.extend(geojson["features"])

        print(f"{tile}: px={int(binary_native.sum()):,}, polygons={len(geojson['features'])}")

    submission = {
        "type": "FeatureCollection",
        "features": all_features,
    }

    out_path = SUB_DIR / f"{out_name}.geojson"

    with open(out_path, "w") as f:
        json.dump(submission, f)

    print(f"Saved: {out_path}")
    print(f"Total polygons: {len(all_features)}")
    return out_path


In [ ]:
if RUN_ENSEMBLE_SUBMISSIONS:
    make_ensemble_submission(
        tag_a=XGB_ENSEMBLE_TAG,
        tag_b=LGB_FINAL_TAG,
        out_name="submission_hybrid_xgb_lgb",
        mode="hybrid",
    )

    make_ensemble_submission(
        tag_a=XGB_ENSEMBLE_TAG,
        tag_b=LGB_FINAL_TAG,
        out_name="submission_hybrid_xgb_lgb_weakfilter",
        mode="hybrid_filter",
    )


## 11. Feature Importance + Run Metadata


In [ ]:
def save_lgb_feature_importance(model, tag):
    if model is None:
        print(f"No LightGBM model for {tag}")
        return None

    imp = pd.DataFrame({
        "feature": names,
        "gain": model.feature_importance(importance_type="gain"),
        "split": model.feature_importance(importance_type="split"),
    }).sort_values("gain", ascending=False)

    out_path = MODEL_DIR / f"feature_importance_{tag}.csv"
    imp.to_csv(out_path, index=False)
    display(imp.head(30))
    print(f"Saved: {out_path}")
    return out_path


def save_xgb_feature_importance(model, tag):
    if model is None:
        print(f"No XGBoost model for {tag}")
        return None

    score = model.get_score(importance_type="gain")
    imp = pd.DataFrame(
        [{"feature": k, "gain": v} for k, v in score.items()]
    ).sort_values("gain", ascending=False)

    out_path = MODEL_DIR / f"feature_importance_{tag}.csv"
    imp.to_csv(out_path, index=False)
    display(imp.head(30))
    print(f"Saved: {out_path}")
    return out_path


def save_run_metadata():
    metadata = {
        "root": str(ROOT),
        "pseudo_gt_dir": str(PGT_DIR),
        "years": YEARS,
        "target_shape": TARGET_SHAPE,
        "n_aef_bands": N_AEF_BANDS,
        "n_features": N_FEATURES,
        "cv_holdout": CV_HOLDOUT,
        "neg_ratio": NEG_RATIO,
        "random_seed": RANDOM_SEED,
        "thresholds": THRESHOLDS,
        "lgb_num_boost_round": LGB_NUM_BOOST_ROUND,
        "xgb_num_boost_round": XGB_NUM_BOOST_ROUND,
        "lgb_params": lgb_params,
        "xgb_params": xgb_params,
        "feature_names": names,
    }

    if "X_kept" in globals():
        metadata["sampled_matrix_shape"] = list(X_kept.shape)
        metadata["sampled_positive_rate"] = float((y_kept == 1).mean())
        metadata["tiles"] = sorted(set(tiles_kept.tolist()))

    out_path = META_DIR / "final_aef_lgbm_xgb_submission_metadata.json"
    with open(out_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved: {out_path}")
    return out_path


save_lgb_feature_importance(lgb_final_model, "lgb_alltiles")
save_lgb_feature_importance(lgb_val_model, "lgb_holdout")
save_xgb_feature_importance(xgb_val_model, "xgb_holdout")
save_xgb_feature_importance(xgb_final_model, "xgb_alltiles")
save_run_metadata()
